# Flight Delay Prediction using Apache Spark (PySpark) and MLlib

**End-to-End Enterprise Big Data & Machine Learning Pipeline**  
Author: Big Data & ML Engineering Team  
Dataset: 550,000+ Flight Records adhering to US Department of Transportation (DOT) Schema  

---

## 1. Problem Statement & Motivation
Flight delays cost the aviation industry over \$30 billion annually in operating costs, passenger re-accommodation, and airspace congestion. Under standard Federal Aviation Administration (FAA) criteria, a flight is officially classified as **delayed** if its actual arrival time is more than **15 minutes** past the scheduled arrival time (`ARR_DELAY > 15.0`).

In this project, Apache Spark serves as the **core computational engine** across every stage of the pipeline:
1. **Distributed Data Ingestion**: Strict explicit `StructType` schema parsing across 550,000+ flight records.
2. **Spark SQL & DataFrame Transformations**: Temporal decomposition, route metrics, and hub traffic features.
3. **Spark Window Functions**: Calculating rolling hourly airport congestion and cumulative daily flight volume.
4. **Parquet Persistence**: Columnar, snappy-compressed, partitioned storage.
5. **Spark MLlib Pipelines**: Multi-stage feature transformations (`StringIndexer`, `OneHotEncoder`, `VectorAssembler`, `StandardScaler`).
6. **Multi-Model Training & Tuning**: 3-fold `CrossValidator` and `ParamGridBuilder` comparing **Logistic Regression**, **Random Forest**, and **Gradient-Boosted Trees (GBT)** with class imbalance weighting.
7. **Diagnostic Visualizations & Structured Streaming**: In-memory Spark aggregations and live micro-batch scoring.

## 2. Environment Setup & Centralized SparkSession
We initialize an optimized PySpark `SparkSession` configured for local multi-core laptop execution.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root in python search path
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Set worker python executable
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from src.spark_session import get_spark_session

spark = get_spark_session("FlightDelayPipelineNotebook", master="local[*]", driver_memory="4g")
print(f"Spark Version: {spark.version}")
print(f"Active Python: {sys.executable}")

## 3. Data Ingestion with Explicit StructType Schema
We load the raw 550,000-record dataset using an explicit `StructType` schema to enforce type safety and optimal parsing speed.

In [ ]:
from src.ingestion import load_raw_flight_data, FLIGHTS_RAW_SCHEMA

raw_df = load_raw_flight_data(spark, csv_path="../data/raw/flights_raw.csv")
print(f"Total Rows: {raw_df.count():,}")
raw_df.printSchema()
raw_df.select("FL_DATE", "OP_CARRIER", "ORIGIN", "DEST", "CRS_DEP_TIME", "DEP_DELAY", "ARR_DELAY", "CANCELLED").show(5)

## 4. Data Cleaning & Feature Engineering
We execute:
- Cleaning: Filter cancelled (`CANCELLED==0`) and diverted (`DIVERTED==0`) flights; drop missing delay records.
- Target creation: `is_delayed = when(ARR_DELAY > 15.0, 1.0).otherwise(0.0)`.
- Temporal extraction: `CRS_DEP_HOUR`, `DAY_OF_WEEK`, `MONTH`, `SEASON`, `DEP_TIME_OF_DAY`.
- Spark SQL Temp View: Carrier delay benchmark rates.
- Spark Window Functions: Hourly origin airport congestion (`ORIGIN_HOURLY_TRAFFIC`) and cumulative departures (`ORIGIN_DAILY_CUMULATIVE_FLIGHTS`).

In [ ]:
from src.preprocessing import clean_flight_data, engineer_features

cleaned_df = clean_flight_data(raw_df)
features_df = engineer_features(spark, cleaned_df)

print(f"Processed Records: {features_df.count():,}")
features_df.show(5, truncate=False)

## 5. Parquet Dataset Inspection
The processed feature set is stored in monthly partitioned Parquet format in `data/processed/flights_processed.parquet`.

In [ ]:
parquet_df = spark.read.parquet("../data/processed/flights_processed.parquet")
print("Class Distribution of Target ('is_delayed'):")
parquet_df.groupBy("is_delayed").count().show()

# Check schema of engineered dataset
parquet_df.printSchema()

## 6. Machine Learning Pipeline with MLlib
We construct an end-to-end Spark ML Pipeline:
1. `StringIndexer`: Encodes categoricals into indices (`OP_CARRIER`, `ORIGIN`, `DEST`, `DEP_TIME_OF_DAY`, `SEASON`, `DISTANCE_GROUP`).
2. `OneHotEncoder`: Expands indexed columns to sparse indicator vectors.
3. `VectorAssembler`: Merges encoded vectors with numerical features.
4. `StandardScaler`: Normalizes numerical features for gradient-based convergence.
5. Inverse Class Weights: Mitigates class imbalance via `weightCol`.

In [ ]:
from src.train import add_class_weights, build_preprocessing_pipeline

weighted_df, w_pos, w_neg = add_class_weights(parquet_df)
print(f"Inverse Class Weights: Positive (Delayed) = {w_pos:.4f} | Negative (On-Time) = {w_neg:.4f}")

prep_stages, assembler = build_preprocessing_pipeline()
print(f"Number of Preprocessing Pipeline Stages: {len(prep_stages)}")

## 7. Model Evaluation & Comparison Results
We load the evaluation metrics and model comparison table generated across 108,146 held-out test flights for:
- Logistic Regression
- Random Forest Classifier
- Gradient-Boosted Trees (GBT)

In [ ]:
import pandas as pd

summary_df = pd.read_csv("../outputs/model_summary.csv")
summary_df.sort_values(by="AUC_ROC", ascending=False)

## 8. Diagnostic Visualizations
We display key analytical and model diagnostic charts generated using distributed Spark aggregations.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].imshow(mpimg.imread("../outputs/figures/delay_by_airline.png"))
axes[0, 0].axis('off')
axes[0, 0].set_title("Carrier Delay Rates", fontsize=13, fontweight='bold')

axes[0, 1].imshow(mpimg.imread("../outputs/figures/delay_by_hour.png"))
axes[0, 1].axis('off')
axes[0, 1].set_title("Hourly Compounding Delay Rates", fontsize=13, fontweight='bold')

axes[1, 0].imshow(mpimg.imread("../outputs/figures/model_comparison.png"))
axes[1, 0].axis('off')
axes[1, 0].set_title("MLlib Model Comparison", fontsize=13, fontweight='bold')

axes[1, 1].imshow(mpimg.imread("../outputs/figures/confusion_matrix.png"))
axes[1, 1].axis('off')
axes[1, 1].set_title("Best Model Confusion Matrix", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Real-Time Inference on New Flight Input
We load the persisted `PipelineModel` from `models/best_flight_delay_model` and score sample flights.

In [ ]:
from src.predict import load_model, prepare_flight_features, predict_flight

model = load_model("../models/best_flight_delay_model")

# Test: Evening Flight from Chicago (ORD) to Los Angeles (LAX) in December
flight_df = prepare_flight_features(
    spark,
    carrier="AA",
    origin="ORD",
    dest="LAX",
    dep_time_hhmm=1830,
    month=12,
    day_of_month=22,
    day_of_week=5,
    distance=1745.0,
    carrier_delay_rate=0.37,
    origin_hourly_traffic=42.0,
    dest_hourly_traffic=30.0,
    origin_daily_cumulative=130.0
)

pred, delay_p, ontime_p = predict_flight(model, flight_df)
status = "DELAYED (>15 min)" if pred == 1 else "ON-TIME"
print(f"Prediction Status: {status}")
print(f"Delay Probability: {delay_p*100:.1f}%")
print(f"On-Time Probability: {ontime_p*100:.1f}%")

## 10. Summary & Production Architecture Conclusions

### Why Apache Spark?
1. **Distributed Memory & Catalyst Optimizer**: Spark processes 550,000+ flight records in seconds, utilizing logical and physical plan optimization, predicate pushdown, and columnar compression.
2. **Tungsten Engine**: Direct off-heap memory management and whole-stage code generation maximize CPU and memory efficiency without Python GIL bottleneck.
3. **Unified ML Pipeline**: Clean feature transformation (`StringIndexer`, `OneHotEncoder`, `VectorAssembler`, `StandardScaler`) serialized into a single portable artifact for both batch training and real-time streaming.
4. **Structured Streaming**: Seamlessly extends the same trained ML model to process streaming live flight events in micro-batches.

Spark session shutdown:

In [ ]:
spark.stop()
print("SparkSession stopped successfully.")